<a href="https://colab.research.google.com/github/cris73tian/TPF-IA-PlantVillage/blob/main/TEST2_TPF_IA_Plant_Village.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importación de Librerias

In [20]:
import os
import random
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report

import kagglehub

# 1. Configuración de semillas para reproducibilidad global
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 2. Descarga del dataset utilizando kagglehub
print("Iniciando descarga automatizada desde Kaggle...")
path_dataset = kagglehub.dataset_download("tushar5harma/plant-village-dataset-updated")
print(f"Dataset listo en la ruta: {path_dataset}")

Iniciando descarga automatizada desde Kaggle...
Using Colab cache for faster access to the 'plant-village-dataset-updated' dataset.
Dataset listo en la ruta: /kaggle/input/plant-village-dataset-updated


Al centralizar las dependencias evita errores de alcance (scope) y reimportaciones. Fijar la semilla (SEED = 42) en todas las librerías estocásticas garantiza que la partición de datos y los pesos iniciales del baseline sean 100% reproducibles. kagglehub resuelve la ingesta de datos directa sin necesidad de gestionar tokens de API manualmente ni depender de archivos locales.  
Se establece un entorno de ejecución estandarizado e inmune a pérdidas de sesión, garantizando la disponibilidad inmediata de los archivos de imagen en el disco virtual.

## Ingesta del Dataset y Extracción de Familias (Group Split por UUID)

In [21]:
data = []

# Escaneo recursivo del directorio descargado
for root, dirs, files_list in os.walk(path_dataset):
    for file in files_list:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(root, file)
            class_name = os.path.basename(root)
            data.append({
                'filepath': full_path,
                'filename': file,
                'class_folder': class_name
            })

df = pd.DataFrame(data)

# Extraer el UUID/Familia base a partir del nombre del archivo ({UUID}_{transformacion}.jpg)
df['family_id'] = df['filename'].apply(lambda x: x.split('_')[0])

# Definir la etiqueta de clase directa desde el nombre de la carpeta contenedora
df['target_class'] = df['class_folder']

print(f"Total de imágenes procesadas: {len(df)}")
print(f"Total de familias/UUIDs únicos: {df['family_id'].nunique()}")
print(f"Total de clases detectadas: {df['target_class'].nunique()}")

Total de imágenes procesadas: 67118
Total de familias/UUIDs únicos: 31496
Total de clases detectadas: 17


El dataset contiene múltiples imágenes que son variaciones sintéticas (rotaciones, espejados) de un mismo espécimen vegetal. Si se usara una división aleatoria estándar, variantes de una misma hoja quedarían repartidas entre conjuntos, produciendo Data Leakage. La extracción de family_id (UUID) es la base para agrupar todas las variantes de una misma hoja dentro de un solo split.  
Queda en evidencia cuantitativamente que el volumen total de archivos es superior al número real de hojas fotografiadas (UUIDs únicos), redefiniendo la verdadera cantidad de muestras independientes disponibles.

## Partición Estratificada por Grupos (80% Train, 10% Dev, 10% Test)

In [22]:
# Primera división: 90% (Train + Dev) y 10% Test
sgkf_outer = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
train_dev_idx, test_idx = next(sgkf_outer.split(df, df['target_class'], groups=df['family_id']))

df_train_dev = df.iloc[train_dev_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

# Segunda división sobre el 90%: 80% Train y 10% Dev (1/9 de train_dev)
sgkf_inner = StratifiedGroupKFold(n_splits=9, shuffle=True, random_state=SEED)
train_idx, dev_idx = next(sgkf_inner.split(df_train_dev, df_train_dev['target_class'], groups=df_train_dev['family_id']))

df_train = df_train_dev.iloc[train_idx].reset_index(drop=True)
df_dev = df_train_dev.iloc[dev_idx].reset_index(drop=True)

print(f"Muestras en Train: {len(df_train)} ({len(df_train)/len(df):.1%})")
print(f"Muestras en Dev:   {len(df_dev)} ({len(df_dev)/len(df):.1%})")
print(f"Muestras en Test:  {len(df_test)} ({len(df_test)/len(df):.1%})")

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 7 members, which is less than n_splits=10.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 7 members, which is less than n_splits=9.
  warnings.warn(


Muestras en Train: 54231 (80.8%)
Muestras en Dev:   6475 (9.6%)
Muestras en Test:  6412 (9.6%)


Se aplica StratifiedGroupKFold anidado en dos niveles para cumplir las exigencias de la Sección 4.1.2. Garantiza dos condiciones fundamentales:  
Aislamiento por UUID: Toda una familia pertenece exclusivamente a Train, Dev o Test.  
Estratificación: Mantiene las proporciones relativas de las clases detectadas a lo largo de los tres subconjuntos.  
Formaliza la creación de los tres splits independientes. El conjunto Dev actúa como una prueba rigurosa de generalización real sobre especímenes nunca observados.

## Auditoría Anti-Leakage y Control de Proporciones (EDA)

In [23]:
# 1. Auditoría programática de intersección de familias
intersection_train_dev = set(df_train['family_id']).intersection(set(df_dev['family_id']))
intersection_train_test = set(df_train['family_id']).intersection(set(df_test['family_id']))
intersection_dev_test = set(df_dev['family_id']).intersection(set(df_test['family_id']))

assert len(intersection_train_dev) == 0, "Error: Leakage entre Train y Dev"
assert len(intersection_train_test) == 0, "Error: Leakage entre Train y Test"
assert len(intersection_dev_test) == 0, "Error: Leakage entre Dev y Test"

print("Auditoría Anti-Leakage aprobada: 0 UUIDs compartidos entre las particiones.")

# 2. Verificación de la distribución proporcional por clase
dist_train = df_train['target_class'].value_counts(normalize=True)
dist_dev = df_dev['target_class'].value_counts(normalize=True)
dist_test = df_test['target_class'].value_counts(normalize=True)

df_distribucion = pd.DataFrame({'Train %': dist_train, 'Dev %': dist_dev, 'Test %': dist_test}).fillna(0)
print("\nDistribución relativa por clase (Primeras 5 clases):")
print(df_distribucion.head())

Auditoría Anti-Leakage aprobada: 0 UUIDs compartidos entre las particiones.

Distribución relativa por clase (Primeras 5 clases):
                     Train %     Dev %    Test %
target_class                                    
.ipynb_checkpoints  0.000111  0.000154  0.000000
Apple Scab          0.038354  0.035830  0.032439
Bacterial Spot      0.099703  0.112587  0.105895
Black Rot           0.071398  0.071351  0.079538
Cedar Apple Rust    0.033044  0.025946  0.037430


Aplicamos controles mediante aserciones que abortan la ejecución en caso de filtración de datos. La tabla de distribución confirma la efectividad del proceso de estratificación por grupo.  
Valida científicamente la partición. Se demuestra que los conjuntos Dev y Test conservan la misma representación porcentual por enfermedad que el conjunto de entrenamiento.

## Dataset PyTorch, DataLoaders y Preprocesamiento Mínimo

In [24]:
# Mapeo de clases a índices numéricos
unique_classes = sorted(df['target_class'].unique())
class_to_idx = {cls_name: i for i, cls_name in enumerate(unique_classes)}

class PlantVillageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['filepath']
        label_str = self.df.iloc[idx]['target_class']
        label = class_to_idx[label_str]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

# Preprocesamiento mínimo (Etapa 1): Reducción a 64x64 px y aplanado vectorial
baseline_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: torch.flatten(x)) # 64 * 64 * 3 = 12288 entradas
])

# Instanciación de DataLoaders
train_dataset = PlantVillageDataset(df_train, transform=baseline_transform)
dev_dataset = PlantVillageDataset(df_dev, transform=baseline_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)

Se cumple con la Sección 4.1.3 (preprocesamiento mínimo). Redimensionar a $64 x 64 y estructurar en vectores 1D de 12.288 dimensiones permite entrenar un modelo denso de baja complejidad rápidamente como baseline, reservando el procesamiento convolucional para la Etapa 3.  
Se genera la canalización (pipeline) de datos optimizada en memoria para alimentar la red durante la etapa inicial de entrenamiento y validación.

## Modelo Baseline (Red Densa Mínima) y Evaluación de Métrica de Número Único

In [25]:
class BaselineMLP(nn.Module):
    def __init__(self, input_dim=12288, hidden_dim=128, num_classes=len(unique_classes)):
        super(BaselineMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_baseline = BaselineMLP().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_baseline.parameters(), lr=0.001)

def evaluate_dev(model, data_loader, device):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    f1_macro = f1_score(all_targets, all_preds, average='macro')
    acc = accuracy_score(all_targets, all_preds)
    return f1_macro, acc

# Entrenamiento del Baseline por 5 épocas
epochs = 5
for epoch in range(epochs):
    model_baseline.train()
    running_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model_baseline(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    dev_f1, dev_acc = evaluate_dev(model_baseline, dev_loader, device)

    print(f"Época [{epoch+1}/{epochs}] - Loss Train: {epoch_loss:.4f} | Dev F1-Macro: {dev_f1:.4f} | Dev Acc: {dev_acc:.4f}")

Época [1/5] - Loss Train: 1.7228 | Dev F1-Macro: 0.3701 | Dev Acc: 0.5095
Época [2/5] - Loss Train: 1.3157 | Dev F1-Macro: 0.3964 | Dev Acc: 0.5592
Época [3/5] - Loss Train: 1.1835 | Dev F1-Macro: 0.4445 | Dev Acc: 0.5893
Época [4/5] - Loss Train: 1.1027 | Dev F1-Macro: 0.4197 | Dev Acc: 0.5955
Época [5/5] - Loss Train: 1.0418 | Dev F1-Macro: 0.5120 | Dev Acc: 0.6073


Al implementa una red densa de 1 capa oculta tal como exige la pauta de la Etapa 1. Evalúa la métrica sobre el conjunto Dev. Se adopta el F1-Score Macro como la métrica de número único principal (Single-Number Evaluation Metric), pues pondera de forma equivalente a todas las categorías sin verse afectada por el desbalance de volumen entre clases.  
Se establece el desempeño inicial computable (Baseline Score) sobre el conjunto Dev. Esta cifra será el punto de partida técnico para medir la evolución en las Etapas 2 y 3.  